In [1]:
import os
from pathlib import Path
from typing import List, Optional, Dict, Tuple
import pickle
import numpy as np
from itertools import combinations, product
from typing import List, Tuple
import pandas as pd
import matplotlib.pyplot as plt



# =============================================================================
# Config (absolute paths + environment)
# =============================================================================
REPO_ROOT = Path(os.getcwd()).resolve().parents[2]
print(REPO_ROOT)

# Experiment selection
MAZES = ['LM4', 'LM6', 'LM8', 'LM8D', 'LMO8', 'LMO8D', 'LM8_addition', 'LMO8_remove', 'BR']
MAZE_INDEX = 4  # LM8
MAZE_KEY = MAZES[MAZE_INDEX]

# Network sizes to evaluate
N_PCS_LIST = [10, 25, 50, 75, 100, 250, 500, 750]

# Output dir
OUT_DIR = REPO_ROOT / "data" / "results" / "wall_metrics" / MAZE_KEY
OUT_DIR.mkdir(parents=True, exist_ok=True)

/Users/titonka/FAIRIS


In [2]:
def _pick_any_df_key(d: Dict[int, pd.DataFrame], preferred_order: Optional[List[int]] = None) -> int:
    if preferred_order:
        for k in preferred_order:
            if k in d:
                return k
    return sorted(d.keys())[0]

def load_xy_and_activations(repo_root: Path, maze_key: str) -> Tuple[np.ndarray, Dict[int, pd.DataFrame]]:
    """Load (x,y) once (from any available N), and the dict N -> DataFrame."""
    act_path = repo_root / "data" / "activations" / f"activations_{maze_key}_test.pkl"
    if not act_path.exists():
        raise FileNotFoundError(f"Missing activations file: {act_path}")
    with open(act_path, "rb") as fh:
        by_n: Dict[int, pd.DataFrame] = pickle.load(fh)
    n_key = _pick_any_df_key(by_n, preferred_order=N_PCS_LIST)
    df0 = by_n[n_key]
    if not {"x", "y"}.issubset(df0.columns):
        raise ValueError(f"(x,y) columns not found in activations dataframe for N={n_key}")
    xy = df0[["x", "y"]].to_numpy(dtype=np.float32)
    return xy, by_n



In [3]:
# Load once using your helper:
xy, by_n = load_xy_and_activations(REPO_ROOT, MAZE_KEY)

In [4]:
# ============================================================
# WALL DEFINITIONS
# ============================================================

# Internal box walls
BOX_WALLS = [
    ((0.75, -1.25), (-0.75, -1.25)),  # W1 bottom
    ((0.75,  1.25), (-0.75,  1.25)),  # W2 top
    ((1.25,  0.75), ( 1.25, -0.75)),  # W3 right
    ((-1.25, 0.75), (-1.25, -0.75)),  # W4 left
]

# Internal X walls
X_WALLS = [
    ((0.50, -0.50), (-0.50,  0.50)),  # W5
    ((-0.50,-0.50), ( 0.50,  0.50)),  # W6
]

# Outer octagon walls
OUTER_WALLS = [
    ((2.61, 1.85), (1.85, 0.00)),
    ((1.85, 2.61), (0.00, 1.85)),
    ((0.00, 1.85), (-1.85, 2.61)),
    ((-1.85, 0.00), (-2.61, 1.85)),
    ((-2.61, 0.00), (-1.85, -1.85)),
    ((-1.85, -1.85), (-0.00, -2.61)),
    ((-0.00, -2.61), (1.85, -1.85)),
    ((1.85, -1.85), (2.61, 0.00)),
]

# Pocket points around X
POCKET_POINTS = [
    (0.0,  0.5),
    (0.0, -0.5),
    (-0.5, 0.0),
    (0.5,  0.0),
]



In [5]:
def point_to_segment_distance(px, py, ax, ay, bx, by):
    AP = np.array([px - ax, py - ay])
    AB = np.array([bx - ax, by - ay])
    denom = np.dot(AB, AB)
    if denom == 0:
        return np.linalg.norm(AP)
    t = max(0, min(1, np.dot(AP, AB) / denom))
    closest = np.array([ax, ay]) + t * AB
    return np.linalg.norm(np.array([px, py]) - closest)

def wall_side(px, py, ax, ay, bx, by):
    v = np.array([bx - ax, by - ay])
    perp = np.array([-v[1], v[0]])
    rel = np.array([px - ax, py - ay])
    s = np.dot(rel, perp)
    if abs(s) < 1e-6:
        return 0
    return 1 if s > 0 else -1


In [6]:
def annotate_with_regions(df, box_walls, pocket_points,
                          wall_max_dist=0.40,
                          pocket_max_dist=0.30):
    """
    Adds:
        wall_id, wall_dist, wall_side
        pocket_id, pocket_dist
        region_type ∈ {"wall_side", "pocket", "none"}
        region_id   ∈ {"W1_inside", "W4_outside", "P2", "none"}
    """
    df = df.copy()
    px = df["x"].to_numpy()
    py = df["y"].to_numpy()
    n = len(df)

    # Wall distances
    wall_ids    = np.full(n, None)
    wall_dists  = np.full(n, np.inf)
    wall_sides  = np.zeros(n)

    for w_id, (A, B) in enumerate(box_walls, start=1):
        ax, ay = A
        bx, by = B

        dists = np.array([
            point_to_segment_distance(px[i], py[i], ax, ay, bx, by)
            for i in range(n)
        ])

        sides = np.array([
            wall_side(px[i], py[i], ax, ay, bx, by)
            for i in range(n)
        ])

        mask = dists < wall_dists
        wall_dists[mask] = dists[mask]
        wall_ids[mask]   = w_id
        wall_sides[mask] = sides[mask]

    df["wall_id"]   = wall_ids
    df["wall_dist"] = wall_dists
    df["wall_side"] = wall_sides

    # Pocket distances
    pocket_ids = np.full(n, None)
    pocket_dists = np.full(n, np.inf)

    for pid, (cx, cy) in enumerate(pocket_points):
        d = np.sqrt((px - cx)**2 + (py - cy)**2)
        mask = d < pocket_dists
        pocket_dists[mask] = d[mask]
        pocket_ids[mask]   = pid

    df["pocket_id"] = pocket_ids
    df["pocket_dist"] = pocket_dists

    # Determine region
    region_type = []
    region_id = []

    for i in range(n):
        dw = df.loc[i, "wall_dist"]
        dp = df.loc[i, "pocket_dist"]

        # Pocket region
        if dp <= pocket_max_dist:
            pid = df.loc[i,"pocket_id"]
            region_type.append("pocket")
            region_id.append(f"P{pid}")

        # Wall-side region
        elif dw <= wall_max_dist:
            side = df.loc[i,"wall_side"]
            side_name = "inside" if side > 0 else "outside"
            w = df.loc[i,"wall_id"]
            region_type.append("wall_side")
            region_id.append(f"W{w}_{side_name}")

        else:
            region_type.append("none")
            region_id.append("none")

    df["region_type"] = region_type
    df["region_id"] = region_id

    return df


In [7]:
def plot_region_assignments(df_annot, box_walls, pocket_points, title="Region Assignments"):
    plt.figure(figsize=(7,7))
    
    regions = sorted(df_annot["region_id"].unique())
    cmap = plt.cm.get_cmap("tab20", len(regions))

    for idx, r in enumerate(regions):
        dfr = df_annot[df_annot["region_id"] == r]
        plt.scatter(dfr["x"], dfr["y"], s=10, color=cmap(idx), label=f"{r} ({len(dfr)})")

    # Draw walls
    for (A, B) in box_walls:
        plt.plot([A[0], B[0]], [A[1], B[1]], "k-", linewidth=3)

    # Pocket centers
    for pid, (cx, cy) in enumerate(pocket_points):
        plt.scatter([cx],[cy], color="black", marker="x", s=80)

    plt.legend(bbox_to_anchor=(1.05,1), loc="upper left")
    plt.title(title)
    plt.gca().set_aspect("equal","box")
    plt.grid(True)
    plt.show()


In [8]:
def get_topk(vec, k):
    idx = np.argpartition(-vec, k-1)[:k]
    idx = idx[np.argsort(-vec[idx])]
    return set(idx.tolist())

def compute_topk_sets(df, pc_cols, k_prop=0.10):
    N = len(pc_cols)
    k = max(1, int(round(k_prop * N)))
    acts = df[pc_cols].to_numpy()
    df = df.copy()
    df["topk"] = [get_topk(acts[i], k) for i in range(len(df))]
    return df, k


In [9]:
def compute_WEAI(df_annot, pc_cols, k_prop=0.10):
    """
    WEAI = Wall-Exclusive Activation Index
    Computes a per-wall and mean WEAI score.

    WEAI(w) = 1 - average_fraction_of_shared_cells_across_wall

    Returns (wall_df, WEAI_global)
    """

    df, k = compute_topk_sets(df_annot, pc_cols, k_prop=k_prop)

    results = []

    for w in [1,2,3,4]:
        inside_label  = f"W{w}_inside"
        outside_label = f"W{w}_outside"

        df_in  = df[df["region_id"] == inside_label]
        df_out = df[df["region_id"] == outside_label]

        if len(df_in)==0 or len(df_out)==0:
            continue

        overlaps = []
        for a,b in product(df_in["topk"], df_out["topk"]):
            overlaps.append(len(a & b) / k)   # normalized shared cells

        mean_overlap = np.mean(overlaps)
        WEAI_w = 1 - mean_overlap

        results.append({"wall_id": w, "WEAI_w": WEAI_w})

    if len(results)==0:
        return pd.DataFrame(), None

    wall_df = pd.DataFrame(results)
    WEAI_global = wall_df["WEAI_w"].mean()

    return wall_df, WEAI_global


In [14]:
all_results = []

for N, dfN in by_n.items():
    print(f"Processing N={N}")

    # Annotate each point with wall/pocket region
    dfN_annot = annotate_with_regions(dfN, BOX_WALLS, POCKET_POINTS)

    pc_cols = [c for c in dfN.columns if c.startswith("pc_")]

    wall_df, WEAI_value = compute_WEAI(dfN_annot, pc_cols, k_prop=0.20)

    all_results.append({
        "N": N,
        "WEAI": WEAI_value
    })

df_WEAI = pd.DataFrame(all_results).sort_values("N")
df_WEAI

Processing N=10
Processing N=25
Processing N=50
Processing N=75
Processing N=100
Processing N=250
Processing N=500
Processing N=750


,N,WEAI
0,10,0.958333
1,25,0.966667
2,50,0.922222
3,75,0.879630
4,100,0.912500
5,250,0.912222
6,500,0.901944
7,750,0.893333
